# Fetching data at warp speed

In [1]:
# 1. Install dependencies
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

# --- CONFIGURATION ---
# NOTE: If you want to keep files permanently, mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# OUT_DIR = Path("/content/drive/MyDrive/data/processed")

OUT_DIR = Path("./code/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["AUD/USD", "NZD/USD"]
MONTHS = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512",
    "202601", "202602", "202603"
]

# PARALLELISM: Use -1 to use all cores, or 4-8 to avoid getting rate-limited
N_JOBS = 4 

# --- HELPER FUNCTIONS ---
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """Function to be executed in parallel with existence check and retries"""
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"⏭️ Skipped (Already Exists): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"⚠️ Empty: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"✅ Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"❌ FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"

# --- EXECUTION ---
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"🚀 Starting Parallel Download of {len(jobs)} jobs using {N_JOBS} workers...\n")

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

for r in results:
    print(r)

elapsed = time.time() - t0
print(f"\n✨ All done! Finished in {elapsed:.1f}s")
print(f"📁 Files saved to: {OUT_DIR.absolute()}")

🚀 Starting Parallel Download of 30 jobs using 4 workers...



INFO:DUKASCRIPT:current timestamp :2025-08-01T12:39:05.982000
INFO:DUKASCRIPT:current timestamp :2025-09-02T01:06:14.486000
INFO:DUKASCRIPT:current timestamp :2025-10-01T12:01:12.875000
INFO:DUKASCRIPT:current timestamp :2025-08-01T14:39:19.766000
INFO:DUKASCRIPT:current timestamp :2025-11-03T16:19:25.743000
INFO:DUKASCRIPT:current timestamp :2025-09-02T12:03:20.508000
INFO:DUKASCRIPT:current timestamp :2025-08-03T22:57:19.376000
INFO:DUKASCRIPT:current timestamp :2025-09-02T23:00:04.488000
INFO:DUKASCRIPT:current timestamp :2025-10-01T16:55:10.372000
INFO:DUKASCRIPT:current timestamp :2025-11-04T08:21:47.539000
INFO:DUKASCRIPT:current timestamp :2025-09-03T14:04:20.350000
INFO:DUKASCRIPT:current timestamp :2025-08-04T13:34:11.295000
INFO:DUKASCRIPT:current timestamp :2025-10-02T12:01:25.650000
INFO:DUKASCRIPT:current timestamp :2025-11-04T18:24:41.608000
INFO:DUKASCRIPT:current timestamp :2025-09-04T07:00:38.142000
INFO:DUKASCRIPT:current timestamp :2025-08-05T06:29:23.107000
INFO:DUK

⏭️ Skipped (Already Exists): AUD/USD 202501
⏭️ Skipped (Already Exists): AUD/USD 202502
⏭️ Skipped (Already Exists): AUD/USD 202503
⏭️ Skipped (Already Exists): AUD/USD 202504
⏭️ Skipped (Already Exists): AUD/USD 202505
⏭️ Skipped (Already Exists): AUD/USD 202506
⏭️ Skipped (Already Exists): AUD/USD 202507
✅ Success: AUD/USD 202508 -> audusd_dukascopy_bid_202508.parquet (989,678 rows) & audusd_dukascopy_ask_202508.parquet (989,678 rows)
✅ Success: AUD/USD 202509 -> audusd_dukascopy_bid_202509.parquet (1,071,817 rows) & audusd_dukascopy_ask_202509.parquet (1,071,817 rows)
✅ Success: AUD/USD 202510 -> audusd_dukascopy_bid_202510.parquet (1,271,762 rows) & audusd_dukascopy_ask_202510.parquet (1,271,762 rows)
✅ Success: AUD/USD 202511 -> audusd_dukascopy_bid_202511.parquet (1,145,408 rows) & audusd_dukascopy_ask_202511.parquet (1,145,408 rows)
✅ Success: AUD/USD 202512 -> audusd_dukascopy_bid_202512.parquet (1,096,710 rows) & audusd_dukascopy_ask_202512.parquet (1,096,710 rows)
✅ Success: 